# Chạy Qwen3-8B (như bài báo) trên Colab GPU, graph trong Neo4j Aura

Notebook này chạy toàn bộ thí nghiệm bằng Qwen3-8B (Ollama, bản Q8) trên GPU của Colab; graph đọc từ Neo4j Aura (`--backend neo4j`).

**Chuẩn bị một lần:**
1. Ở máy: `scripts/pack_colab.sh` → tải `outputs/colab_bundle.tar.gz` lên Google Drive, thư mục `MyDrive/KhoaLuan/`.
2. Tạo Neo4j AuraDB Free ở <https://console.neo4j.io> (nên chọn vùng GCP `us-central1`, gần máy Colab). Lưu URI và mật khẩu.
3. Trong Colab, mục 🔑 **Secrets** (thanh bên trái), thêm `NEO4J_URI` (dạng `neo4j+s://xxxx.databases.neo4j.io`), `NEO4J_USER` (`neo4j`), `NEO4J_PASSWORD`, và bật quyền truy cập cho notebook. Notebook không in các giá trị này.
4. Runtime → Change runtime type → **T4 GPU** (hoặc L4/A100 nếu có).

**Dừng giữa chừng không sao:** cache các lời gọi LLM nằm trên Drive (`MyDrive/KhoaLuan/cache_llm`); chạy lại từ đầu notebook thì các câu đã xong được lấy từ cache, không gọi lại.

Phần chấm kết luận bằng LLM (`--no-grade` ở đây) làm sau ở máy bằng deepseek, để mọi lượt chạy dùng cùng một người chấm.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/KhoaLuan'
import os
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)
os.makedirs(f'{DRIVE}/cache_llm', exist_ok=True)

In [ ]:
BRANCH = 'claude/legalgraphrag-framework-hv23z3'
%cd /content
!rm -rf Agent-Graph-Rag && git clone -q -b {BRANCH} https://github.com/bighousevn/Agent-Graph-Rag.git
%cd /content/Agent-Graph-Rag
!pip install -q -r requirements.txt
!tar xzf {DRIVE}/colab_bundle.tar.gz
!ls -la outputs/hierargraph.pkl data/processed/cases_vn_test.json data/raw/questions.xlsx

## Biến môi trường (không in ra)

In [ ]:
from google.colab import userdata
for k in ('NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD'):
    os.environ[k] = userdata.get(k)

MODEL = 'qwen3:8b-q8_0'  # Qwen3-8B như bài báo; Q8 vừa T4 16 GB cùng ngữ cảnh 16k
os.environ.update({
    'LLM_PROVIDER': 'ollama',
    'LLM_API_KEY': 'ollama',          # Ollama không cần key; biến này chỉ để chọn đúng nhà cung cấp
    'LLM_BASE_URL': 'http://localhost:11434/v1',
    'LLM_MODEL': MODEL,
    'LLM_TIMEOUT': '600',
    'CACHE_DIR': f'{DRIVE}/cache_llm',
})
print('Đã đặt biến môi trường cho', MODEL)

## Ollama + Qwen3-8B

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, urllib.request
env = {**os.environ, 'OLLAMA_CONTEXT_LENGTH': '16384', 'OLLAMA_KEEP_ALIVE': '-1'}
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434'); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}

In [ ]:
# Thử một lời gọi: phải ra JSON, không có <think>
import sys; sys.path.insert(0, '.')
from vn_legal_graph.config import AppConfig
from vn_legal_graph.llm import LLMClient
c = LLMClient(AppConfig.from_env_file('/nonexistent'))
c.config.cache_llm_calls = False
t = time.time()
print(c.generate('Trả về JSON {"toi_danh": [...]} cho hành vi: lén lấy xe máy của hàng xóm.'), f'({time.time() - t:.1f}s)')

## Đưa graph lên Neo4j Aura

Chỉ cần chạy **một lần** (xoá dữ liệu cũ trong database rồi ghi lại). Cuối cùng in kết quả so sánh với graph trong RAM; phải giống hoàn toàn.

In [ ]:
EXPORT = True  # đổi thành False ở các lần chạy sau
if EXPORT:
    !python scripts/export_neo4j.py
else:
    from vn_legal_graph.graph.neo4j_store import Neo4jGraph, connect
    print(Neo4jGraph(connect()).stats())

## Chạy thí nghiệm

Mỗi ô chạy xong thì kết quả được chép sang `MyDrive/KhoaLuan/outputs/`. Thời gian là ước lượng trên T4.

In [ ]:
import glob, shutil
def save():
    for f in glob.glob('outputs/*.json'):
        shutil.copy(f, f'{DRIVE}/outputs/')
    print(sorted(os.listdir(f'{DRIVE}/outputs')))

### 1. Hỏi–đáp: Qwen tự trả lời, không graph (187 câu, ~187 lời gọi, ~15 phút)

In [ ]:
!python scripts/run_qa_pilot.py --auto-gold --n 1000 --without-graph --no-grade --backend neo4j --tag full 2>&1 | grep -v '^  '
save()

### 2. Hỏi–đáp: pipeline graph (187 câu, ≤20 lời gọi/câu, vài giờ)

In [ ]:
!python scripts/run_qa_pilot.py --auto-gold --n 1000 --rerank --no-grade --backend neo4j --tag full 2>&1 | grep -v '^  '
save()

### 3. Vụ án: Qwen tự kết luận, không graph (55 án)

In [ ]:
!python scripts/run_case_pipeline.py --without-graph --backend neo4j 2>&1 | tail -15
save()

### 4. Vụ án: Researcher → Auditor (lọc cứng) → Adjudicator, như bài gốc (55 án, vài giờ)

In [ ]:
!python scripts/run_case_pipeline.py --auditor loc --backend neo4j --tag loc 2>&1 | tail -15
save()

### 5. Vụ án: bỏ Auditor, Adjudicator xem thêm án tương tự (cấu hình tốt nhất với deepseek)

In [ ]:
!python scripts/run_case_pipeline.py --auditor bo-qua --adjudicator-context an --backend neo4j --tag boqua_an 2>&1 | tail -15
save()

## Sau khi chạy xong (ở máy)

Chép các file `qa_*_qwen3_8b-q8_0_full.json` và `case_*_qwen3_8b-q8_0*.json` từ Drive về `outputs/`, rồi chấm kết luận hỏi–đáp bằng deepseek:

```
python scripts/run_qa_pilot.py --grade-only outputs/qa_llm_only_qwen3_8b-q8_0_full.json
python scripts/run_qa_pilot.py --grade-only outputs/qa_pilot_qwen3_8b-q8_0_full.json
```

Điểm vụ án (accuracy, F1) không cần LLM chấm, đã có sẵn trong file.